In [5]:
cd "C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph"

C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph


# Evaluation function


# Testing Evaluation Function

In [13]:
from src.evaluation.metrics import EvaluationMetrics

In [14]:
reference_summary = (
    "The team discussed improving website performance and database indexing."
)

generated_summary = (
    "The meeting focused on website performance improvements and database optimization."
)

rouge_scores = EvaluationMetrics.calculate_rouge(
    reference_summary,
    generated_summary,
)

rouge_scores

{'rouge1': 0.631578947368421,
 'rouge2': 0.23529411764705882,
 'rougeL': 0.5263157894736842}

In [15]:
bert_score = EvaluationMetrics.calculate_bertscore(
    reference_summary,
    generated_summary,
)

bert_score

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


0.9505724310874939

# Hallucination Validator

# test validation.py

In [28]:
from src.evaluation.validator import HallucinationValidator

transcript = """
John: We need to improve website performance.

Sarah: I will optimize frontend assets this week.

David: I will improve database indexing.
"""

generated_decisions = [
    "Sarah: I will optimize frontend assets this week.",
    "David: I will improve database indexing.",
    "Alice will deploy the mobile application."
]

validation_results = HallucinationValidator.validate_statements(
    transcript,
    generated_decisions,
)

print(validation_results)

{'Sarah: I will optimize frontend assets this week.': True, 'David: I will improve database indexing.': True, 'Alice will deploy the mobile application.': False}


# End-to-End Evaluation Notebook

## import

In [29]:
from pathlib import Path

from src.config.config import load_config
from pathlib import Path

from src.config.config import load_config
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker

from src.vector_store.embedding_model import EmbeddingModel
from src.vector_store.faiss_index import FAISSIndexManager
from src.vector_store.retriever import TranscriptRetriever

from src.graph.workflow import MeetingWorkflow
from src.graph.meeting_state import MeetingState

from src.evaluation.metrics import EvaluationMetrics
from src.evaluation.validator import HallucinationValidator

## Run Complete Workflow

In [31]:
config=load_config()

TRANSCRIPT_PATH = (Path(config["paths"]["raw_data"]) /"Meeting_Transcript.txt")

document=TranscriptLoader.load_document(str(TRANSCRIPT_PATH))

clean_document=TranscriptCleaner.clean(document)

nodes=TranscriptChunker.create_nodes(clean_document)

embedding_model=EmbeddingModel.load_model()

vector_index= FAISSIndexManager.create_index( nodes=nodes,embedding_model=embedding_model,)

retriever=TranscriptRetriever.create_retriever(vector_index)

state: MeetingState = {
    "retriever": retriever,
    "topics": None,
    "summary": None,
    "action_items": None,
    "priorities": None,
}

workflow = MeetingWorkflow.build()

result = workflow.invoke(state)



2026-09-11 13:42:38 | INFO | src.data_ingestion.loader | Transcript loaded successfully: Meeting_Transcript.txt
2026-09-11 13:42:38 | INFO | src.preprocessing.cleaner | Transcript Normalized Sucessfully
2026-09-11 13:42:38 | INFO | src.preprocessing.chunker | Created 2 transcript chunks.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-09-11 13:42:47 | INFO | src.vector_store.embedding_model | Embedding model loaded: BAAI/bge-small-en-v1.5
2026-09-11 13:42:49 | INFO | src.vector_store.faiss_index | FAISS vector index created successfully.
2026-09-11 13:42:49 | INFO | src.vector_store.retriever | Retriever created with Top-K = 3
2026-09-11 13:42:49 | INFO | src.graph.workflow | meeting workflow compiled successfully
2026-09-11 13:43:02 | INFO | src.llm.llm | Loaded LLM: Qwen2.5-3B-Instruct.gguf
2026-09-11 13:43:02 | INFO | src.agents.topic_agent | Topic Agent executed successfully.
2026-09-11 13:44:12 | INFO | src.graph.graph_nodes | Topic node Completed
2026-09-11 13:44:15 | INFO | src.llm.llm | Loaded LLM: Qwen2.5-3B-Instruct.gguf
2026-09-11 13:44:15 | INFO | src.agents.summary_agent | Summary Agent executed successfully.
2026-09-11 13:45:32 | INFO | src.graph.graph_nodes | Summary node completed.
2026-09-11 13:45:35 | INFO | src.llm.llm | Loaded LLM: Qwen2.5-3B-Instruct.gguf
2026-09-11 13:45:35 | INFO | src.ag

In [32]:
result

{'retriever': <llama_index.core.indices.vector_store.retrievers.retriever.VectorIndexRetriever at 0x138ebd52f90>,
 'topics': TopicOutput(topics=['login issue', 'API response format', 'error handling in mobile app', 'authentication API changes', 'device-specific login issues']),
 'summary': SummaryOutput(meeting_objective='Identify and resolve the mobile app crashing issue during login.', key_discussion_points=['The root cause of the crash is related to authentication API changes deployed three days ago.', 'A detailed bug report with screenshots and logs has been prepared by QA Tester.', 'Mobile Developer reviewed Android login module for error handling, while Backend Developer verified authentication API for breaking changes.'], decisions_taken=['QA Tester will share the bug report within the next hour.', 'Mobile Developer will start working on the fix immediately after this meeting.', 'Backend Developer will review server logs right away to identify any authentication errors recorded.

# Reference Summary

In [33]:
reference_summary = """
The meeting focused on improving website performance, database indexing,
and assigning frontend optimization tasks. The team agreed on immediate
performance improvements and database optimization work.
"""

In [34]:
generated_summary = result["summary"].meeting_objective + "\n\n"

generated_summary += "\n".join(
    result["summary"].key_discussion_points
)

generated_summary += "\n\n"

generated_summary += "\n".join(
    result["summary"].decisions_taken
)

print(generated_summary)

Identify and resolve the mobile app crashing issue during login.

The root cause of the crash is related to authentication API changes deployed three days ago.
A detailed bug report with screenshots and logs has been prepared by QA Tester.
Mobile Developer reviewed Android login module for error handling, while Backend Developer verified authentication API for breaking changes.

QA Tester will share the bug report within the next hour.
Mobile Developer will start working on the fix immediately after this meeting.
Backend Developer will review server logs right away to identify any authentication errors recorded.


## ROUGE Evaluation

In [35]:
rouge_scores = EvaluationMetrics.calculate_rouge(
    reference_summary,
    generated_summary,
)

rouge_scores

{'rouge1': 0.13333333333333333, 'rouge2': 0.0, 'rougeL': 0.08333333333333333}

## BERTScore Evaluation

In [36]:
bert_score = EvaluationMetrics.calculate_bertscore(
    reference_summary,
    generated_summary,
)

bert_score

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


0.835752010345459

In [45]:
ground_truth_tasks = [
    "Review the Android login module ",
    "Prepare a detailed bug report",
]

In [46]:
predicted_tasks = [
    item.task
    for item in result["action_items"].action_items
]

In [48]:
all_tasks = sorted(
    list(
        set(ground_truth_tasks + predicted_tasks)
    )
)

y_true = [
    1 if task in ground_truth_tasks else 0
    for task in all_tasks
]

y_pred = [
    1 if task in predicted_tasks else 0
    for task in all_tasks
]

precision_recall = EvaluationMetrics.calculate_precision_recall(
    y_true,
    y_pred,
)

precision_recall

{'precision': 0.0, 'recall': 0.0}

In [44]:
print("Ground Truth:")
print(ground_truth_tasks)

print("\nPredicted:")
print(predicted_tasks)

Ground Truth:
['Optimize frontend assets', 'Improve database indexing']

Predicted:
['Review the Android login module and check if any error handling is missing.', 'Prepare a detailed bug report with screenshots and logs.', 'Verify the authentication API and check if there are any breaking changes.']


In [40]:
transcript_text = clean_document.text

In [41]:
generated_tasks = [
    item.task
    for item in result["action_items"].action_items
]

validation_results = HallucinationValidator.validate_statements(
    transcript_text,
    generated_tasks,
)

validation_results

{'Review the Android login module and check if any error handling is missing.': True,
 'Prepare a detailed bug report with screenshots and logs.': True,
 'Verify the authentication API and check if there are any breaking changes.': True}

In [42]:
evaluation_report = {
    "rouge": rouge_scores,
    "bert_score": round(bert_score, 4),
    "precision": round(
        precision_recall["precision"], 4
    ),
    "recall": round(
        precision_recall["recall"], 4
    ),
    "hallucination_check": validation_results,
}

evaluation_report

{'rouge': {'rouge1': 0.13333333333333333,
  'rouge2': 0.0,
  'rougeL': 0.08333333333333333},
 'bert_score': 0.8358,
 'precision': 0.0,
 'recall': 0.0,
 'hallucination_check': {'Review the Android login module and check if any error handling is missing.': True,
  'Prepare a detailed bug report with screenshots and logs.': True,
  'Verify the authentication API and check if there are any breaking changes.': True}}

In [43]:
print("=" * 50)
print("MEETING NOTES EVALUATION REPORT")
print("=" * 50)

print("\nROUGE Scores")
for metric, value in rouge_scores.items():
    print(f"{metric}: {value:.4f}")

print(f"\nBERTScore: {bert_score:.4f}")

print("\nAction Item Evaluation")
print(f"Precision: {precision_recall['precision']:.4f}")
print(f"Recall   : {precision_recall['recall']:.4f}")

print("\nHallucination Validation")
for task, status in validation_results.items():
    label = "SUPPORTED" if status else "NOT SUPPORTED"
    print(f"- {task}: {label}")

MEETING NOTES EVALUATION REPORT

ROUGE Scores
rouge1: 0.1333
rouge2: 0.0000
rougeL: 0.0833

BERTScore: 0.8358

Action Item Evaluation
Precision: 0.0000
Recall   : 0.0000

Hallucination Validation
- Review the Android login module and check if any error handling is missing.: SUPPORTED
- Prepare a detailed bug report with screenshots and logs.: SUPPORTED
- Verify the authentication API and check if there are any breaking changes.: SUPPORTED
